# Global-level Connectivity — Confirmatory Analysis 

## Purpose

This notebook executes the **confirmatory pipeline** for global connectivity-derived depression subtypes. It validates the structure-function coupling cluster structure produced by `06_global_clustering.ipynb` by re-testing — via quantile regression — whether the identified subtypes (Cluster 0 and Cluster 1) differ from Controls and from each other in terms of global connectivity, after adjusting for age, sex, head motion and selected ICD-10 comorbidities.

The analysis is repeated independently for three connectivity modalities:

| Modality | Description |
|---|---|
| `functional` | Resting-state fMRI functional connectivity |
| `structural` | Diffusion MRI structural connectivity |
| `sfc` | Structure-Function Coupling (Pearson *r* between FC and SC node profiles) |

---

## Notebook structure

| # | Section | What happens |
|---|---|---|
| 1 | Imports & configuration | Set all paths, column names, and analysis parameters |
| 2 | Output-capture helpers | `Tee` class and `capture_stdout_to_log` context manager for logging |
| 3 | Step 0 — R environment init | Load `quantreg` once via `rpy2` |
| 4 | Step 1 — Load cohort data | Read `global_merged_connectivity_clusters.csv`; inspect shape and group breakdown |
| 5 | Step 2 — Per-modality QR | For each connectivity type: run 3-model quantile regression in R and display raw p-values |
| 6 | Step 3 — FDR correction | Apply Benjamini-Hochberg FDR to the 4 contrasts; display corrected p-values |
| 7 | Step 4 — Violin plot | Create and save 4-group violin with significance brackets |
| 8 | Outputs summary | Verify all expected output files were written to disk |

---

## Expected inputs

The following files must exist before running this notebook:

| File | Description |
|---|---|
| `data/UKB/cohorts/global_merged_connectivity_clusters.csv` | Wide-format cohort table produced by `global_clustering_main.py`. Must include columns `eid`, `Group`, `{ct}_connectivity` (one per modality), `sfc_cluster`, `p21003_i2` (age), `p31` (sex), `p24441_i2` (fMRI motion), `p24453_i2` (dMRI motion), and optional ICD indicator columns (`I10`, `Z864`, `F419`) |

---

## Outputs

| Output | Path |
|---|---|
| Per-modality run log | `data/UKB/F32_notask_STRCO_RSSCHA_RSTIA/global_{ct}_connectivity_output_sfc_clusters.txt` |
| Per-modality FDR log | `data/UKB/F32_notask_STRCO_RSSCHA_RSTIA/global_{ct}_connectivity_FDR_sfc_clusters.txt` |
| Violin plot PNG | `reports/plots/schaefer1000+tian54/sfc_con/{ct}_global_conn_sfc_confirmatory_violinplots.png` |

> **Note:** The R regression helper writes a temporary CSV to `/tmp/combined_data.csv` during each call. Avoid running modalities in parallel to prevent file collisions.

---

## Runtime requirements

- Python: `numpy`, `pandas`, `matplotlib`, `seaborn`, `sklearn`, `statsmodels`, `rpy2`
- System R with packages: `quantreg`, `multcomp` (auto-installed by the utils if missing)

## Section 1 — Imports & Configuration

Import all required Python packages and set the module-level constants that control which modalities are analysed, which ICD covariates are included as confounders, where data lives, and where outputs are written.

Key constants:

| Constant | Value / meaning |
|---|---|
| `CONNECTIVITY_TYPES` | `['functional', 'structural', 'sfc']` — modalities to loop over |
| `ICD_COVARIATES` | `['I10', 'Z864', 'F419']` — comorbidity columns used as QR covariates |
| `CLUSTER_COL` | `'sfc_cluster'` — column carrying the subtype label (from SFC-derived clustering) |
| `GROUP_COL` | `'Group'` — Control / Depression label column |
| `QUANTILE_REGRESSION_BOOTSTRAP_R` | `1000` — bootstrap iterations for R's `boot.rq` SE estimation |

In [ ]:
import os
import sys
import io
from contextlib import contextmanager

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Add the project source to the Python path so the cluster utilities are importable
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
SRC_ROOT = os.path.join(PROJECT_ROOT, 'source_code')
if SRC_ROOT not in sys.path:
    sys.path.insert(0, SRC_ROOT)

from clusters.global_clustering_confirmatory_utils import (
    setup_r_environment,
    apply_multiple_testing_correction,
    get_motion_columns,
    run_quantile_regression,
    create_violin_plot_with_significance,
)

# ── Connectivity modalities ────────────────────────────────────────────────────
CONNECTIVITY_TYPES = ['functional', 'structural', 'sfc']

# ── ICD-10 comorbidity indicators (most common in the depression cohort) ───────
ICD_COVARIATES = ['I10', 'Z864', 'F419']

# ── Cohort column names ────────────────────────────────────────────────────────
CLUSTER_COL = 'sfc_cluster'   # subtype label column (from SFC-derived clustering)
GROUP_COL   = 'Group'          # 'Control' / 'Depression'

# ── Motion metric column names ─────────────────────────────────────────────────
fMRI_MOTION_METRIC = 'p24441_i2'   # fMRI framewise displacement
dMRI_MOTION_METRIC = 'p24453_i2'   # dMRI mean relative head motion

# ── Bootstrap replications for quantile regression SE estimation ───────────────
QUANTILE_REGRESSION_BOOTSTRAP_R = 5000

# ── File-system paths ──────────────────────────────────────────────────────────
DEPRESSION_DIR          = os.path.join(PROJECT_ROOT, 'data/UKB/F32_notask_STRCO_RSSCHA_RSTIA')
COHORTS_DIR             = os.path.join(PROJECT_ROOT, 'data/UKB/cohorts')
VALIDATION_PLOTS_BASE_DIR = os.path.join(PROJECT_ROOT, 'reports/plots/schaefer1000+tian54')

MERGED_COHORT_PATH = os.path.join(COHORTS_DIR, 'global_merged_connectivity_clusters.csv')

print("Configuration loaded.")
print(f"  Modalities    : {CONNECTIVITY_TYPES}")
print(f"  ICD covariates: {ICD_COVARIATES}")
print(f"  Cluster column: {CLUSTER_COL}")
print(f"  Group column  : {GROUP_COL}")
print(f"  Bootstrap R   : {QUANTILE_REGRESSION_BOOTSTRAP_R}")

## Section 2 — Output-capture helpers

The main script uses a small `Tee` class and a `capture_stdout_to_log` context manager to mirror everything printed to stdout into a per-modality log file on disk. We replicate these helpers here so the notebook can optionally persist console output to the same log paths used by the script.

In [ ]:
class Tee(io.TextIOBase):
    """Write to both the original stdout and a StringIO buffer simultaneously.

    Used so that R output (printed to stdout by rpy2) is captured alongside
    normal Python print statements and persisted to a log file.
    """
    def __init__(self, original, buffer):
        self.original = original
        self.buffer   = buffer

    def write(self, s):
        self.original.write(s)
        self.buffer.write(s)
        return len(s)

    def flush(self):
        self.original.flush()
        self.buffer.flush()


@contextmanager
def capture_stdout_to_log(log_path):
    """Context manager: mirror stdout to a buffer and write buffer to *log_path* on exit.

    Usage
    -----
    ```python
    with capture_stdout_to_log('/path/to/run.log'):
        # everything printed here is also saved to disk
        ...
    ```
    """
    os.makedirs(os.path.dirname(log_path), exist_ok=True)
    original_stdout = sys.stdout
    buffer = io.StringIO()
    sys.stdout = Tee(original_stdout, buffer)
    try:
        yield buffer
    finally:
        sys.stdout = original_stdout
        with open(log_path, 'w', encoding='utf-8') as f:
            f.write(buffer.getvalue())

print("Output-capture helpers defined.")

## Step 0 — Initialize R environment

Before any quantile regression can run, we need to ensure the R packages `quantreg` and `multcomp` are available in the system R installation. `setup_r_environment()` (from the utils module) calls `install.packages()` for each package if it is not already present, then imports them via `rpy2`.

This step only needs to run **once** per kernel session.

In [ ]:
print("Initializing R environment ...")
r_env = setup_r_environment()
print("R packages loaded successfully:")
for pkg_name in r_env:
    print(f"  ✓ {pkg_name}")

## Step 1 — Load cohort data

We read the single wide-format cohort CSV produced by `global_clustering_main.py`. This table is the key input to the confirmatory pipeline: it holds per-subject scalar connectivity summaries and subtype labels for all three modalities in a single file.

**Expected columns (minimum set):**

| Column | Description |
|---|---|
| `eid` | Subject identifier |
| `Group` | `'Control'` or `'Depression'` |
| `functional_connectivity` | Scalar summary of resting-state FC per subject |
| `structural_connectivity` | Scalar summary of dMRI SC per subject |
| `sfc_connectivity` | Scalar summary of SFC per subject |
| `sfc_cluster` | Subtype label (`'Cluster 0'` / `'Cluster 1'` for depressed; `'Control'` for controls) |
| `p21003_i2` | Age at imaging visit (continuous) |
| `p31` | Biological sex (categorical) |
| `p24441_i2` | fMRI mean relative head motion |
| `p24453_i2` | dMRI mean relative head motion |
| `I10`, `Z864`, `F419` | ICD-10 comorbidity indicators (binary 0/1) |

In [ ]:
assert os.path.isfile(MERGED_COHORT_PATH), (
    f"Merged cohort CSV not found at:\n  {MERGED_COHORT_PATH}\n"
    "Run global_clustering_main.py first to generate this file."
)

ALL_COMBINED = pd.read_csv(MERGED_COHORT_PATH)
print(f"Loaded: {MERGED_COHORT_PATH}")
print(f"  Shape: {ALL_COMBINED.shape[0]} subjects × {ALL_COMBINED.shape[1]} columns")

# ── Group breakdown ────────────────────────────────────────────────────────────
print("\nGroup breakdown:")
print(ALL_COMBINED[GROUP_COL].value_counts().to_string())

# ── Cluster breakdown (SFC-derived labels) ─────────────────────────────────────
if CLUSTER_COL in ALL_COMBINED.columns:
    print(f"\nCluster label breakdown ('{CLUSTER_COL}'):")
    print(ALL_COMBINED[CLUSTER_COL].value_counts().to_string())
else:
    print(f"\n[WARNING] Column '{CLUSTER_COL}' not found in the loaded table.")

# ── Column presence check ──────────────────────────────────────────────────────
required_cols = (
    ['eid', GROUP_COL, CLUSTER_COL,
     'p21003_i2', 'p31', fMRI_MOTION_METRIC, dMRI_MOTION_METRIC]
    + [f'{ct}_connectivity' for ct in CONNECTIVITY_TYPES]
    + ICD_COVARIATES
)
missing = [c for c in required_cols if c not in ALL_COMBINED.columns]
if missing:
    print(f"\n[WARNING] Missing expected columns: {missing}")
else:
    print("\nAll required columns present ✓")

ALL_COMBINED.head(3)

## Step 2 — Prepare output directories

Ensure all output directories exist before entering the per-modality loop. The main script creates them implicitly; in the notebook we make this explicit.

In [ ]:
validation_plots_dir = os.path.join(VALIDATION_PLOTS_BASE_DIR, 'sfc_con')
os.makedirs(DEPRESSION_DIR, exist_ok=True)
os.makedirs(validation_plots_dir, exist_ok=True)

print(f"Run log directory   : {DEPRESSION_DIR}")
print(f"Violin plot directory: {validation_plots_dir}")

## Step 3 — Per-modality confirmatory pipeline

The next cells iterate over each connectivity modality and perform the following sub-steps:

### Step 3.1 — Quantile regression (R)

`run_quantile_regression()` fits three separate quantile regression models at the median (τ = 0.5) in R using `quantreg::rq()` with bootstrap SE estimation (`se = "boot", R = 1000`). Each model includes age (mean-centred), sex, head-motion (mean-centred per modality), and ICD-10 comorbidity indicators as covariates.

The three models and the four contrasts returned are:

| Model | Groups compared | Contrast key in result dict |
|---|---|---|
| M1 | Depression vs Control | `depression_vs_control` |
| M2 | Cluster 0 vs Control | `cluster0_vs_control` |
| M2 | Cluster 1 vs Control | `cluster1_vs_control` |
| M3 | Cluster 0 vs Cluster 1 (ref) | `cluster0_vs_cluster1` |

**Inputs to `run_quantile_regression`:**

| Parameter | Value used here |
|---|---|
| `combined_df` | `ALL_COMBINED` (the merged wide table) |
| `conn_type` | Current modality token (`'functional'`, `'structural'`, `'sfc'`) |
| `icd_covariates` | `ICD_COVARIATES` |
| `motion_covariates` | Modality-specific list from `get_motion_columns()` |
| `tau` | `0.5` (median) |
| `R` | `QUANTILE_REGRESSION_BOOTSTRAP_R` (1000) |
| `dependent_var` | `'{ct}_connectivity'` |
| `cluster_col` | `CLUSTER_COL` (`'sfc_cluster'`) |
| `group_col` | `GROUP_COL` (`'Group'`) |

**Returns:** `{'dependent_var': str, 'p_values': dict}` — four raw p-values, one per contrast.

---

### Step 3.2 — FDR correction

`apply_multiple_testing_correction()` applies Benjamini-Hochberg FDR to the four raw p-values. It also appends a formatted table to a per-modality log file.

**Returns:** `(reject_array, corrected_pvalues_array)`.

---

### Step 3.3 — Violin plot

`create_violin_plot_with_significance()` creates a four-group violin (Control, Depression, Cluster 0, Cluster 1) with min-max normalised connectivity values and significance brackets annotated above each pair of groups.

**Saved to:** `<VALIDATION_PLOTS_BASE_DIR>/sfc_con/{ct}_global_conn_sfc_confirmatory_violinplots.png`

In [ ]:
# Store results from all modalities for later inspection
all_results = {}

for conn_type in CONNECTIVITY_TYPES:
    display_conn = (
        "Structure-Function Coupling" if conn_type == "sfc" else conn_type.capitalize()
    )
    print(f"\n{'='*72}")
    print(f"  MODALITY: {display_conn}  ({conn_type})")
    print(f"{'='*72}")

    # ── Per-modality output paths ──────────────────────────────────────────────
    general_log_path = os.path.join(
        DEPRESSION_DIR, f'global_{conn_type}_connectivity_output_sfc_clusters.txt'
    )
    fdr_log_path = os.path.join(
        DEPRESSION_DIR, f'global_{conn_type}_connectivity_FDR_sfc_clusters.txt'
    )
    violin_out_path = os.path.join(
        validation_plots_dir,
        f'{conn_type}_global_conn_sfc_confirmatory_violinplots.png'
    )

    # ── Motion covariates for this modality ────────────────────────────────────
    # get_motion_columns() returns a dict e.g. {'motion_fmri': 'p24441_i2'}
    # We only pass the column-name values (list) to run_quantile_regression.
    motion_columns       = get_motion_columns(conn_type, fMRI_MOTION_METRIC, dMRI_MOTION_METRIC)
    motion_covariate_list = list(motion_columns.values())
    print(f"  Motion covariates : {motion_covariate_list}")

    # ── Step 3.1 — Quantile regression ────────────────────────────────────────
    print(f"\n  [3.1] Running quantile regression (tau=0.5, R={QUANTILE_REGRESSION_BOOTSTRAP_R} bootstraps)...")

    with capture_stdout_to_log(general_log_path):
        qr_results = run_quantile_regression(
            ALL_COMBINED,
            conn_type,
            ICD_COVARIATES,
            motion_covariates=motion_covariate_list,
            tau=0.5,
            R=QUANTILE_REGRESSION_BOOTSTRAP_R,
            dependent_var=f'{conn_type}_connectivity',
            cluster_col=CLUSTER_COL,
            group_col=GROUP_COL,
        )

    print("  Raw p-values returned from R:")
    for contrast_name, pval in qr_results['p_values'].items():
        print(f"    {contrast_name:<35} p = {pval:.6g}")

    # ── Step 3.2 — FDR correction ──────────────────────────────────────────────
    print(f"\n  [3.2] Applying Benjamini-Hochberg FDR correction ...")
    _, corrected_p_values = apply_multiple_testing_correction(
        p_values=list(qr_results['p_values'].values()),
        test_methods=[
            'QR (median) depression vs control',
            'QR (median) cluster 0 vs control',
            'QR (median) cluster 1 vs control',
            'QR (median) cluster 0 vs cluster 1',
        ],
        variable_names=[f'Global {conn_type.capitalize()} Connectivity'] * 4,
        log_path=fdr_log_path,
        log_context=f'{display_conn} — SFC-cluster confirmatory',
    )

    contrast_names = list(qr_results['p_values'].keys())
    print("  FDR-corrected p-values:")
    for name, p_adj in zip(contrast_names, corrected_p_values):
        sig = '***' if p_adj < 0.001 else '**' if p_adj < 0.01 else '*' if p_adj < 0.05 else 'n.s.'
        print(f"    {name:<35} p_adj = {p_adj:.6g}  {sig}")

    # ── Step 3.3 — Violin plot ─────────────────────────────────────────────────
    print(f"\n  [3.3] Creating violin plot → {violin_out_path}")

    subject_scalar_control    = ALL_COMBINED.loc[ALL_COMBINED[GROUP_COL] == 'Control',    f'{conn_type}_connectivity']
    subject_scalar_depression = ALL_COMBINED.loc[ALL_COMBINED[GROUP_COL] == 'Depression', f'{conn_type}_connectivity']

    # Map text cluster labels to integer 0/1 as required by the plotting helper
    clusters = ALL_COMBINED.loc[ALL_COMBINED[GROUP_COL] == 'Depression', CLUSTER_COL].copy()
    clusters = clusters.replace({'Cluster 0': 0, 'Cluster 1': 1}).astype(int)

    create_violin_plot_with_significance(
        subject_scalar_control,
        subject_scalar_depression,
        clusters,
        corrected_p_values,
        conn_type,
        violin_out_path,
    )

    # ── Store for summary ──────────────────────────────────────────────────────
    all_results[conn_type] = {
        'raw_p_values'      : qr_results['p_values'],
        'corrected_p_values': dict(zip(contrast_names, corrected_p_values.tolist())),
        'violin_path'       : violin_out_path,
        'fdr_log_path'      : fdr_log_path,
        'run_log_path'      : general_log_path,
    }

print(f"\n{'='*72}")
print("All modalities complete.")

## Step 4 — Summarise results across modalities

Collect raw and FDR-corrected p-values for all contrasts and all modalities into a single tidy DataFrame for easy inspection and export.

In [ ]:
summary_rows = []
for ct, res in all_results.items():
    for contrast in res['raw_p_values']:
        p_raw = res['raw_p_values'][contrast]
        p_adj = res['corrected_p_values'].get(contrast, float('nan'))
        sig   = '***' if p_adj < 0.001 else '**' if p_adj < 0.01 else '*' if p_adj < 0.05 else 'n.s.'
        summary_rows.append({
            'modality'    : ct,
            'contrast'    : contrast,
            'p_raw'       : round(p_raw, 6),
            'p_adj (FDR)' : round(p_adj, 6),
            'significance': sig,
        })

summary_df = pd.DataFrame(summary_rows)
print("P-value summary across modalities and contrasts:\n")
print(summary_df.to_string(index=False))
summary_df

## Step 5 — Display saved violin plots inline

Load the PNG files saved during the per-modality loop and display them directly in the notebook for a quick visual review.

In [ ]:
from IPython.display import Image, display

for ct, res in all_results.items():
    vpath = res['violin_path']
    display_name = "Structure-Function Coupling" if ct == "sfc" else ct.capitalize()
    print(f"\n── {display_name} ──")
    if os.path.isfile(vpath):
        display(Image(filename=vpath, width=900))
    else:
        print(f"  [NOT FOUND] {vpath}")

## Step 6 — Verify all output files

Confirm that every expected output file (violin PNGs, run logs, FDR logs) was written to disk.

In [ ]:
print("Output file verification:\n")
all_ok = True
for ct, res in all_results.items():
    checks = {
        'Violin plot PNG' : res['violin_path'],
        'Run log (stdout)': res['run_log_path'],
        'FDR log'         : res['fdr_log_path'],
    }
    for label, path in checks.items():
        exists = os.path.isfile(path)
        status = '✓' if exists else '✗ MISSING'
        if not exists:
            all_ok = False
        size = f"({os.path.getsize(path):,} bytes)" if exists else ''
        print(f"  [{ct:12s}] {label:<22} {status} {size}")
    print()

if all_ok:
    print("All outputs present ✓")
else:
    print("Some outputs are missing — check the cells above for errors.")